In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset


batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(1337)

print(f"Kullanılan Donanım: {device.upper()}")
if device == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

dataset = load_dataset("tascib/turkish-llm-dataset", split="train", streaming=True)

text_tr = ""
target_size_bytes = 1024 * 1024 * 1.5

for i, sample in enumerate(dataset):
    if "text" in sample and sample["text"]:
        text_tr += sample["text"] + "\n"

    if len(text_tr.encode('utf-8')) > target_size_bytes or i > 25000:
        break

print(f"Derlenen Türkçe metin boyutu: {len(text_tr.encode('utf-8')) / 1024:.2f} KB")

chars_tr = sorted(list(set(text_tr)))
vocab_size_tr = len(chars_tr)
print(f"Türkçe Veri Seti - Benzersiz Karakter Sayısı (Vocab Size): {vocab_size_tr}")

stoi_tr = { ch:i for i,ch in enumerate(chars_tr) }
itos_tr = { i:ch for i,ch in enumerate(chars_tr) }
encode_tr = lambda s: [stoi_tr[c] for c in s if c in stoi_tr]
decode_tr = lambda l: ''.join([itos_tr[i] for i in l])

data_tr = torch.tensor(encode_tr(text_tr), dtype=torch.long)
n_tr = int(0.9 * len(data_tr))
train_data_tr = data_tr[:n_tr]
val_data_tr = data_tr[n_tr:]

def get_batch_tr(split):
    d = train_data_tr if split == 'train' else val_data_tr
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss_tr(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_tr(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size_tr, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size_tr)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel().to(device)
print(f"Toplam Parametre Sayısı: {sum(p.numel() for p in model.parameters())/1e6:.2f} Milyon")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)



for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss_tr(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch_tr('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Metin Üretimi
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\nTürkçe Metin Örneği")
print(decode_tr(model.generate(context, max_new_tokens=500)[0].tolist()))

Kullanılan Donanım: CUDA
GPU Model: Tesla T4
Hugging Face üzerinden Türkçe veri seti yükleniyor...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/2.74k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/3383 [00:00<?, ?it/s]

Hedef veri boyutu: ~1.5 MB toplanıyor...
Derlenen Türkçe metin boyutu: 1536.55 KB
Türkçe Veri Seti - Benzersiz Karakter Sayısı (Vocab Size): 95
Model Oluşturuldu. Toplam Parametre Sayısı: 10.81 Milyon
Eğitim başlıyor...
step 0: train loss 4.7448, val loss 4.7458
step 500: train loss 1.6429, val loss 1.7198
step 1000: train loss 1.2792, val loss 1.3975
step 1500: train loss 1.1524, val loss 1.2926
step 2000: train loss 1.0679, val loss 1.2328
step 2500: train loss 1.0092, val loss 1.1939
step 3000: train loss 0.9583, val loss 1.1702
step 3500: train loss 0.9190, val loss 1.1578
step 4000: train loss 0.8782, val loss 1.1405
step 4500: train loss 0.8434, val loss 1.1309
step 4999: train loss 0.8125, val loss 1.1241

--- T4 Modeli ile Üretilen Türkçe Metin Örneği ---

Ankan Batta yaş, sanatçıa, minimal ve musiki esasi 1. 8. yesi bar sanda sayılmaya çalışılan da Talenin Sultundan Hicaz , Arban, Kantemiri Haydn'ın Klarnet Hazır Koydurucugunun , Çine gelir, Ulvid-vila-Lova semahları için, en 